# Calculate region level population data

In [ ]:
import os
import xarray as xr
import numpy as np

In [ ]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"

In [ ]:
# Use region mask to allign population data
mask_file = "GBD_Region_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)

# Adjust indices to match (with small tolerance)
# e.g., max 1e-7 km distance
pop = pop.reindex_like(masks, method="nearest", tolerance=1e-9)

In [ ]:
# Loop over regions, sum the population for each country and apply to list
population_by_region = []
for i in range(len(masks.region)):
    print(masks.isel(region=i)["region"].values)
    mask = masks.isel(region=i)
    region = masks.isel(region=i)["region"]
    pop_region = (xr.where(
        mask == 1,
        pop,
        np.nan)).sum(dim=("lat", "lon"))
    population_by_region.append(pop_region)

pop_array = xr.concat(population_by_region, "region")

In [ ]:
# Save region level population
description = ("Region level population sum for years 2000-2100 "
               "- scripts by A.F. Wells (2025)")

pop_array.attrs["description"] = description

out_file = "ssp2_region_level_2000-2100.nc"
out_path = os.path.join(POP_DIR, out_file)
pop_array.to_netcdf(out_path)

print("All processing complete.")